# NBA Player Roster and Salary Analysis

This notebook explores player counts, team and position composition, age distribution, and salary patterns in the included NBA roster dataset.

> **Data-quality note:** The source `Height` column contains mixed values, including strings that appear to have been converted to dates. These values cannot safely be reconstructed. This notebook preserves the raw height field and excludes it from quantitative analysis rather than inventing measurements.

## 1. Load data and inspect structure

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv(Path("myexcel.csv"))
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
display(df.head())
display(df.isna().sum().rename("missing_values").to_frame())
print("Duplicate rows:", df.duplicated().sum())

## 2. Data preparation

In [ ]:
df = df.drop_duplicates().copy()
df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Height_raw"] = df["Height"]
print(f"Rows after exact duplicate removal: {len(df):,}")
print(f"Missing salary values: {df['Salary'].isna().sum():,}")
print("Height is excluded: source values have inconsistent formats and cannot be reliably parsed.")

## 3. Player distribution by team and position

In [ ]:
team_counts = df["Team"].value_counts()
position_counts = df["Position"].value_counts()

fig, ax = plt.subplots(figsize=(12, 7))
team_counts.sort_values().plot(kind="barh", ax=ax)
ax.set(title="NBA players by team", xlabel="Player count", ylabel="Team")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
position_counts.plot(kind="bar", ax=ax)
ax.set(title="Player distribution by position", xlabel="Position", ylabel="Player count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

display(position_counts.rename("player_count").to_frame())

## 4. Age distribution

In [ ]:
age_data = df.dropna(subset=["Age"])
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=age_data, x="Age", bins=15, kde=True, ax=ax)
ax.set(title="NBA player age distribution", xlabel="Age (years)", ylabel="Player count")
plt.tight_layout()
plt.show()
display(age_data["Age"].describe().to_frame(name="age"))

## 5. Salary analysis

In [ ]:
salary_data = df.dropna(subset=["Salary"]).copy()
print(f"Players with salary recorded: {len(salary_data):,} / {len(df):,}")
display(salary_data["Salary"].describe().to_frame(name="salary"))

team_salary = salary_data.groupby("Team")["Salary"].agg(
    total_salary="sum",
    median_salary="median",
    players_with_salary="count"
).sort_values("total_salary", ascending=False)
display(team_salary.head(10))

fig, ax = plt.subplots(figsize=(12, 7))
team_salary["total_salary"].sort_values().plot(kind="barh", ax=ax)
ax.set(title="Recorded salary total by team", xlabel="Salary (dataset units)", ylabel="Team")
plt.tight_layout()
plt.show()

position_salary = salary_data.groupby("Position")["Salary"].median().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
position_salary.plot(kind="bar", ax=ax)
ax.set(title="Median recorded salary by position", xlabel="Position", ylabel="Median salary (dataset units)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Age and salary relationship

In [ ]:
age_salary = salary_data.dropna(subset=["Age", "Salary"])
pearson_corr = age_salary["Age"].corr(age_salary["Salary"], method="pearson")
spearman_corr = age_salary["Age"].corr(age_salary["Salary"], method="spearman")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"Spearman correlation: {spearman_corr:.3f}")

fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=age_salary, x="Age", y="Salary", hue="Position", alpha=0.75, ax=ax)
ax.set(title="Player age and recorded salary", xlabel="Age (years)", ylabel="Salary (dataset units)")
ax.legend(title="Position", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 7. Interpretation and limitations

Use the tables and charts above to identify team roster sizes, common positions, age distribution and salary patterns.

- Salary comparisons exclude rows with missing salary values.
- Team salary totals may be affected by missing salary records.
- Correlation is descriptive and does not establish causation.
- The height field is intentionally excluded because its source values are inconsistent and corrupted.
- Verify the dataset's original source, snapshot date and license before redistribution.

This is an NBA player dataset, not an employee/workforce dataset.